# Fine-tune PP-OCRv5 mobile rec trên crop biển số Việt Nam

**Chạy thẳng từ mã nguồn đồ án** — mở trong VS Code, chọn kernel Python bất kỳ rồi chạy
tuần tự. Không cần Google Drive: dữ liệu, mã PaddleOCR, bộ trọng số gốc và nơi lưu kết quả
đều nằm trong kho mã.

**Mục tiêu:** nâng độ chính xác chuỗi **biển 2 dòng** (hiện ~0,60 trên tập đánh giá 2.801 mẫu)
bằng cách fine-tune bộ nhận dạng ký tự trên đúng phân phối mà hệ thống thật đưa vào OCR
(strip 2-dòng-ghép-ngang, cao 64 px).

### Cách notebook này làm việc

Mọi lệnh nặng chạy qua **`training-work/venv`** (môi trường riêng đã cài PaddlePaddle 3.3.1),
gọi bằng `subprocess` — nên **kernel của notebook không cần cài gì cả**, và môi trường
`backend/.venv` đang phục vụ hệ thống thật cũng không bị đụng tới.

Mỗi cell kiểm tra thứ nó cần và **tự chuẩn bị nếu thiếu**; chạy lại một cell đã xong là vô hại.

> ⚠️ **Thời gian.** Trên GPU NVIDIA: ~1–2 giờ. **Trên CPU: ~2 giờ mỗi epoch** (đo thật trên
> i5-14600K, 20 luồng) ⇒ 30 epoch mất khoảng **60 giờ**. Cell 6 sẽ nói rõ máy này thuộc trường
> hợp nào; nếu là CPU, hãy đọc phần *Chiến lược khi chỉ có CPU* ở cell 6 trước khi bấm chạy.


In [ ]:
# 1) Xac dinh kho ma va cac duong dan — chay cell nay truoc tien
import json, os, subprocess, sys, urllib.request
from pathlib import Path

# Notebook nam o ai/training/ => goc kho ma la thu muc cha thu hai.
ROOT = Path.cwd()
while not (ROOT / 'CLAUDE.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

WORK       = ROOT / 'training-work'
VENV_PY    = WORK / 'venv' / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
PADDLEOCR  = WORK / 'PaddleOCR'
PRETRAINED = WORK / 'pretrained' / 'en_PP-OCRv5_mobile_rec_pretrained.pdparams'
DATA       = ROOT / 'datasets' / 'processed' / 'rec_finetune'
OUTPUT     = WORK / 'output' / 'rec_vn'
EXPORT     = ROOT / 'models' / 'rec_finetuned'

def run(args, cwd=None):
    """Chay lenh, in log truc tiep ra notebook (khong nuot output)."""
    print('$', ' '.join(str(a) for a in args))
    return subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None).returncode

print('Kho ma      :', ROOT)
print('Du lieu     :', DATA, '|', 'CO' if DATA.exists() else 'CHUA CO')
print('PaddleOCR   :', PADDLEOCR, '|', 'CO' if PADDLEOCR.exists() else 'CHUA CO')
print('Trong so goc:', 'CO' if PRETRAINED.exists() else 'CHUA CO')
print('Venv train  :', 'CO' if VENV_PY.exists() else 'CHUA CO')


In [ ]:
# 2) Moi truong train rieng (PaddlePaddle 3.3.1) — tao neu chua co
#    MOC: in 'paddle 3.3.1 san sang'. Lan dau chay mat vai phut.
if not VENV_PY.exists():
    run([sys.executable, '-m', 'venv', WORK / 'venv'])
    run([VENV_PY, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
    # CPU build; may co GPU NVIDIA thi xem ghi chu duoi cell nay.
    run([VENV_PY, '-m', 'pip', 'install', '-q', 'paddlepaddle==3.3.1'])

out = subprocess.run([str(VENV_PY), '-c',
    'import paddle; print(paddle.__version__, paddle.device.is_compiled_with_cuda())'],
    capture_output=True, text=True)
ver, has_cuda = out.stdout.split() if out.returncode == 0 else ('?', 'False')
USE_GPU = has_cuda == 'True'
print(f'paddle {ver} san sang | CUDA: {USE_GPU}')


**Máy có GPU NVIDIA?** Bản CPU ở trên vẫn chạy được nhưng rất chậm. Muốn dùng GPU, chạy một lần:

```
training-work/venv/Scripts/pip uninstall -y paddlepaddle
training-work/venv/Scripts/pip install paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
```

rồi chạy lại cell 2 — phải thấy `CUDA: True`. *(Cờ `-i` là bắt buộc: bản GPU 3.x không có trên
PyPI, thiếu cờ này pip sẽ báo "No matching distribution".)*


In [ ]:
# 3) Du lieu huan luyen — sinh tu corpus nhan neu chua co
#    MOC: 6672 dong train.txt / 571 dong val.txt.
if not (DATA / 'train.txt').exists():
    print('Chua co dataset, dang sinh (vai phut)...')
    run([ROOT / 'backend' / '.venv' / 'Scripts' / 'python.exe',
         ROOT / 'scripts' / 'dataset' / 'build_rec_finetune_set.py'], cwd=ROOT)

for name in ('train.txt', 'val.txt'):
    n = sum(1 for _ in (DATA / name).open(encoding='utf-8'))
    print(f'{name}: {n} dong')
print('vi du :', (DATA / 'train.txt').open(encoding='utf-8').readline().strip())
print('charset:', (DATA / 'dict36.txt').read_text(encoding='utf-8').split(), '-> 36 ky tu')


In [ ]:
# 4) Ma nguon PaddleOCR (chua tools/train.py) — clone neu chua co
#    MOC: in duong dan config va 'Config OK'.
if not PADDLEOCR.exists():
    WORK.mkdir(exist_ok=True)
    run(['git', 'clone', '--depth', '1',
         'https://github.com/PaddlePaddle/PaddleOCR.git', PADDLEOCR])
    run([VENV_PY, '-m', 'pip', 'install', '-q', '-r', PADDLEOCR / 'requirements.txt'])

CONFIG = PADDLEOCR / 'configs/rec/PP-OCRv5/multi_language/en_PP-OCRv5_mobile_rec.yaml'
if not CONFIG.exists():
    found = list(PADDLEOCR.glob('configs/rec/**/*en_PP-OCRv5_mobile*.y*ml'))
    print('Duong dan mac dinh doi, tim thay:', found)
    CONFIG = found[0]
print('CONFIG =', CONFIG.relative_to(ROOT), '| Config OK')


In [ ]:
# 5) Bo trong so goc en_PP-OCRv5_mobile_rec — tai neu chua co
#    MOC: file khoang 70 MB. Vai tram BYTE nghia la trang loi JSON, KHONG phai model.
URL = ('https://paddle-model-ecology.bj.bcebos.com/paddlex/'
       'official_pretrained_model/en_PP-OCRv5_mobile_rec_pretrained.pdparams')
PRETRAINED.parent.mkdir(parents=True, exist_ok=True)
if not PRETRAINED.exists() or PRETRAINED.stat().st_size < 10_000_000:
    print('Dang tai trong so goc...')
    urllib.request.urlretrieve(URL, PRETRAINED)
size_mb = PRETRAINED.stat().st_size / 1e6
print(f'{PRETRAINED.name}: {size_mb:.1f} MB', '(OK)' if size_mb > 10 else '(SAI — xoa va tai lai)')


### Chiến lược khi chỉ có CPU

Đo thật trên i5-14600K (20 luồng): **~2 giờ/epoch**, tức 30 epoch ≈ 60 giờ. Phiên chạy trước
dừng ở epoch 5 với `acc = 0,166` — đường cong đang lên đều (`norm_edit_dis` 0,37 → 0,81) nhưng
còn xa mức dùng được.

Ba cách xử lý, chọn một:

| Cách | Việc phải làm | Thời gian |
|---|---|---|
| **Chạy nhiều đêm** | để máy thức, cell 6 tự nối tiếp checkpoint cũ mỗi lần chạy lại | ~50 giờ còn lại |
| **Rút số epoch** | sửa `EPOCHS = 12` ở cell 6 — thường đã đủ để thấy xu hướng | ~24 giờ |
| **Mượn GPU** | Colab/Kaggle T4; xem `ai/training/README-rec-finetune.md` | ~1,5 giờ |

**Checkpoint lưu mỗi epoch**, nên dừng lúc nào cũng có model dùng được — và cell 6 luôn tự
tiếp tục từ chỗ dừng chứ không train lại từ đầu.


In [ ]:
# 6) HUAN LUYEN
#    - Tu nhan biet checkpoint cu de TIEP TUC (khong train lai tu dau)
#    - Checkpoint luu moi epoch vao training-work/output/rec_vn/
#    MOC: sau moi 200 iter in 'cur metric, acc: ...' — con so nay phai TANG DAN.
EPOCHS = 30          # CPU: can nhac ha xuong 12 (xem bang o tren)
BATCH  = 128 if USE_GPU else 64
WORKERS = 2 if USE_GPU else 0    # Windows + CPU: 0 worker de tranh treo qua dem

resume = OUTPUT / 'latest.pdparams'
opts = [
    f'Global.use_gpu={str(USE_GPU).lower()}',
    f'Global.character_dict_path={(DATA / "dict36.txt").as_posix()}',
    'Global.use_space_char=false',
    'Global.max_text_length=10',
    f'Global.epoch_num={EPOCHS}',
    'Global.save_epoch_step=1',
    'Global.eval_batch_step=[0,200]',
    'Global.print_batch_step=20',
    f'Global.save_model_dir={OUTPUT.as_posix()}',
    'Optimizer.lr.learning_rate=0.0001',
    'Optimizer.lr.warmup_epoch=1',
    f'Train.dataset.data_dir={DATA.as_posix()}',
    f'Train.dataset.label_file_list=[{(DATA / "train.txt").as_posix()}]',
    f'Train.sampler.first_bs={BATCH}',
    f'Train.loader.batch_size_per_card={BATCH}',
    f'Train.loader.num_workers={WORKERS}',
    f'Eval.dataset.data_dir={DATA.as_posix()}',
    f'Eval.dataset.label_file_list=[{(DATA / "val.txt").as_posix()}]',
    f'Eval.loader.batch_size_per_card={BATCH}',
    f'Eval.loader.num_workers={WORKERS}',
]
if resume.exists():
    print('>> Tiep tuc tu checkpoint:', resume.name)
    opts.insert(1, f'Global.checkpoints={(OUTPUT / "latest").as_posix()}')
else:
    print('>> Bat dau tu trong so goc')
    opts.insert(1, f'Global.pretrained_model={str(PRETRAINED)[:-9]}')

run([VENV_PY, 'tools/train.py', '-c', CONFIG, '-o', *opts], cwd=PADDLEOCR)


### Về cảnh báo `shape ... not matched` lúc bắt đầu

Log sẽ in vài dòng `WARNING: The shape of model params head.ctc_head.fc.weight
paddle.Size([120, 37]) not matched with loaded params ... paddle.Size([120, 438])`.

**Đây là chủ đích, không phải lỗi.** Model gốc có 438 lớp ký tự (tiếng Anh đầy đủ); đồ án dùng
**charset 36 ký tự** (`0-9A-Z`, quyết định Phase 1) nên hai lớp đầu ra được khởi tạo lại còn
backbone vẫn nạp nguyên. Thấy `load pretrain successful` ngay sau đó là đúng.


In [ ]:
# 7) Danh gia checkpoint tot nhat tren tap val sach (571 mau, khong augment)
#    MOC: GHI LAI con so 'acc' — day la so de doi chieu voi baseline.
run([VENV_PY, 'tools/eval.py', '-c', CONFIG, '-o',
     f'Global.use_gpu={str(USE_GPU).lower()}',
     f'Global.checkpoints={(OUTPUT / "best_accuracy").as_posix()}',
     f'Global.character_dict_path={(DATA / "dict36.txt").as_posix()}',
     'Global.use_space_char=false', 'Global.max_text_length=10',
     f'Eval.dataset.data_dir={DATA.as_posix()}',
     f'Eval.dataset.label_file_list=[{(DATA / "val.txt").as_posix()}]',
     f'Eval.loader.batch_size_per_card={BATCH}',
     f'Eval.loader.num_workers={WORKERS}'], cwd=PADDLEOCR)


In [ ]:
# 8) Xuat inference model THANG vao models/rec_finetuned/ cua kho ma
#    MOC: thu muc co cac file inference.* — do la thu ALPR_OCR_REC_MODEL_DIR tro toi.
run([VENV_PY, 'tools/export_model.py', '-c', CONFIG, '-o',
     f'Global.checkpoints={(OUTPUT / "best_accuracy").as_posix()}',
     f'Global.character_dict_path={(DATA / "dict36.txt").as_posix()}',
     'Global.use_space_char=false', 'Global.max_text_length=10',
     f'Global.save_inference_dir={EXPORT.as_posix()}'], cwd=PADDLEOCR)

print()
for f in sorted(EXPORT.glob('*')):
    print(f'  {f.name:<28} {f.stat().st_size/1e6:8.2f} MB')


In [ ]:
# 9) Thu model moi ngay tai day, truoc khi bat cho ca he thong
#    MOC: doc duoc chuoi bien so tu anh demo. So sanh voi model goc o dong duoi.
import os
test_img = ROOT / 'demo' / 'images' / '2dong-1.png'   # bien 2 dong, ground truth 59K1-201.73

snippet = f'''
import sys; sys.path.insert(0, r"{ROOT}")
import cv2
from ai.inference.config import InferenceConfig
from ai.inference.recognizer import PaddleOcrRecognizer
img = cv2.imread(r"{test_img}")
for label, kw in (("model goc     ", {{}}), ("model fine-tune", {{"ocr_rec_model_dir": r"{EXPORT}"}})):
    cfg = InferenceConfig(model_path=r"{ROOT / 'models' / 'best.pt'}", **kw)
    print(label, repr(PaddleOcrRecognizer(cfg).recognize(img).raw_text))
'''
run([ROOT / 'backend' / '.venv' / 'Scripts' / 'python.exe', '-c', snippet], cwd=ROOT)


## Bật model mới cho cả hệ thống

Model đã nằm sẵn ở `models/rec_finetuned/`. Bật bằng **một biến môi trường** — mọi dây nối
đã có sẵn trong mã:

```
# Chạy trực tiếp:  set ALPR_OCR_REC_MODEL_DIR=models/rec_finetuned
# Docker (.env):   ALPR_OCR_REC_MODEL_DIR=/app/models/rec_finetuned
```

Đường dẫn sai sẽ **báo lỗi ngay lúc khởi động** thay vì âm thầm chạy model gốc.

### Đo lại trước khi tin — bắt buộc

Chi tiết ở `ai/training/README-rec-finetune.md`:

1. `ai/evaluation/ocr_accuracy.py` toàn tập, so với baseline trong `docs/reports/16-*`
2. Ba bộ hồi quy: 16 ảnh lõi (`demo/images/expected.json`), 13 ca rescue, 3 video demo
3. Ablation: chạy cả khi bật và khi tắt biến môi trường

**Chỉ tiêu đặt trước:** biển 2 dòng tăng **≥ 5 điểm**, biển 1 dòng **không giảm**. Không đạt thì
gỡ cờ, giữ model gốc, ghi kết quả âm vào báo cáo — một thí nghiệm thất bại có số liệu vẫn là
nội dung tốt cho mục hạn chế của luận văn.
